# Atividade 7

Separar o conjunto de dados breast-cancer-wisconsin.csv em treino, validação e testes. Aplicar dois algoritmos em uma validação cruzada.

Fazer ajustes nos hiperparâmetros dos algoritmos e repetir o processo de validação. Foi possível melhorar os resultados?

Persista o melhor modelo alcançado para ser implantado em um sistema, e crie uma API para acessá-lo em um servidor.

In [31]:
%pip install optuna xgboost fg-data-profiling ipywidgets flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 kB 9.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [35]:
#imports
import pandas as pd
import optuna

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report
from sklearn.metrics import RocCurveDisplay

from sklearn.preprocessing import LabelEncoder

from data_profiling import ProfileReport

import joblib

In [3]:
df = pd.read_csv("breastcancerwisconsin.csv")
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


In [7]:
report = ProfileReport(df,explorative=True, sensitive=True)

In [ ]:
report.to_notebook_iframe() 
# apaguei a saída de célula para enviar no pvanet, mas esse bloco faz uma análise exploratória dos dados

In [12]:
# dados:

def train_validation_test_split(X, y, test_size=0.15, val_size=0.15, random_state=42):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    val_size_ajustado = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_size_ajustado,
        random_state=random_state,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

encoder = LabelEncoder()
X = df.drop(columns=['id', 'diagnosis', 'Unnamed: 32'])
y = encoder.fit_transform(df['diagnosis'])

X_train, X_val, X_test, y_train, y_val, y_test = train_validation_test_split(X, y)

In [53]:
joblib.dump(encoder, "encoder.joblib")

['encoder.joblib']

In [24]:
# treinando modelos sem ajustar hiper-parametros:

def treina_modelo_sem_ajuste(classe_modelo):

    model_XGB = classe_modelo()
    model_XGB.fit(X_train, y_train)

    # Validação cruzada com 5 folds
    scores = cross_val_score(model_XGB, X_val, y_val, cv=5, scoring='accuracy', verbose=1)

    print(f'Acurácia do cross validation: {scores.mean()}')

    print(f'Extraindo métricas do dataset teste:')

    previsoes = model_XGB.predict(X_test)

    print("\nRelatório de Classificação:")
    print(classification_report(y_test, previsoes, digits=5))

In [25]:
treina_modelo_sem_ajuste(XGBClassifier)

Acurácia do cross validation: 0.9294117647058824
Extraindo métricas do dataset teste:

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.94643   0.98148   0.96364        54
           1    0.96667   0.90625   0.93548        32

    accuracy                        0.95349        86
   macro avg    0.95655   0.94387   0.94956        86
weighted avg    0.95396   0.95349   0.95316        86



[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.1s finished


In [26]:
treina_modelo_sem_ajuste(RandomForestClassifier)

Acurácia do cross validation: 0.9529411764705882
Extraindo métricas do dataset teste:

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.94643   0.98148   0.96364        54
           1    0.96667   0.90625   0.93548        32

    accuracy                        0.95349        86
   macro avg    0.95655   0.94387   0.94956        86
weighted avg    0.95396   0.95349   0.95316        86



[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.3s finished


In [32]:
# agora vamos treinar os modelos fazendo ajuste de hiper-parametros

def ajusta_parametros(objective, modelo, nome):
    study = optuna.create_study(direction='maximize')

    study.optimize(objective, n_trials=100)
    print('Melhor acurácia:', study.best_value)
    print('Melhores parâmetros:', study.best_params)

    print("Métricas com modelo e hiper-parametros ajustados")
    model = modelo(**study.best_params)
    model.fit(X_train, y_train)

    print(f'Extraindo métricas do dataset teste:')

    previsoes = model.predict(X_test)
    joblib.dump(model, f"{nome}.joblib")
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, previsoes, digits=5))

def objective_XGB(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    model = XGBClassifier(**params)

    model.fit(X_train, y_train)

  # Validação cruzada com 5 folds
    scores = cross_val_score(model, X_val, y_val, cv=5, scoring='accuracy', verbose=1)
    return scores.mean()

def objective_RF(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ["gini", "entropy", "log_loss"]),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
    }

    model = RandomForestClassifier(**params)

    model.fit(X_train, y_train)

  # Validação cruzada com 5 folds
    scores = cross_val_score(model, X_val, y_val, cv=5, scoring='accuracy', verbose=1)
    return scores.mean()

In [36]:
ajusta_parametros(objective_XGB, XGBClassifier, "xgb")

[I 2026-10-27 22:08:55,816] A new study created in memory with name: no-name-f2345a63-645d-4be5-bf5c-d0339c43cdf1
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.2s finished
[I 2026-10-27 22:08:56,144] Trial 0 finished with value: 0.9529411764705882 and parameters: {'n_estimators': 237, 'max_depth': 3, 'learning_rate': 0.04960920737954483, 'subsample': 0.7039712092037798, 'colsample_bytree': 0.6000632292947001}. Best is trial 0 with value: 0.9529411764705882.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.2s finished
[I 2026-10-27 22:08:56,467] Trial 1 finished with value: 0.9294117647058824 and parameters: {'n_estimators': 239, 'max_depth': 9, 'learning_rate': 0.018273613299608922, 'subsample': 0.7669121749157726, 'colsample_bytree': 0.7685149224505143}. Best is trial 0 with value: 0.9529411764705882.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.4s finished
[I 2026-10-27 22:08:56,935] Trial 2 finished with value: 0.9411764705882353 and parameters: {'n

Melhor acurácia: 0.9764705882352942
Melhores parâmetros: {'n_estimators': 165, 'max_depth': 9, 'learning_rate': 0.08949784952325804, 'subsample': 0.6176988962630369, 'colsample_bytree': 0.5540031541801593}
Métricas com modelo e hiper-parametros ajustados
Extraindo métricas do dataset teste:

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.94643   0.98148   0.96364        54
           1    0.96667   0.90625   0.93548        32

    accuracy                        0.95349        86
   macro avg    0.95655   0.94387   0.94956        86
weighted avg    0.95396   0.95349   0.95316        86



In [37]:
ajusta_parametros(objective_RF, RandomForestClassifier, "rf")

[I 2026-10-27 22:10:11,624] A new study created in memory with name: no-name-2ee235a4-4fdf-4704-a65b-7977d2d0a77b
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.2s finished
[I 2026-10-27 22:10:11,877] Trial 0 finished with value: 0.9294117647058823 and parameters: {'n_estimators': 73, 'bootstrap': False, 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_split': 18, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.9294117647058823.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.2s finished
[I 2026-10-27 22:10:13,457] Trial 1 finished with value: 0.9176470588235294 and parameters: {'n_estimators': 492, 'bootstrap': False, 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_split': 14, 'min_samples_leaf': 13}. Best is trial 0 with value: 0.9294117647058823.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.8s finished
[I 2026-10-27 22:10:14,410] Trial 2 finished with value: 0.9176470588235294 and parameters: {'n_estimators': 243, 'bootstrap

Melhor acurácia: 0.9647058823529413
Melhores parâmetros: {'n_estimators': 150, 'bootstrap': True, 'criterion': 'log_loss', 'max_features': 'log2', 'min_samples_split': 14, 'min_samples_leaf': 6}
Métricas com modelo e hiper-parametros ajustados
Extraindo métricas do dataset teste:

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.94643   0.98148   0.96364        54
           1    0.96667   0.90625   0.93548        32

    accuracy                        0.95349        86
   macro avg    0.95655   0.94387   0.94956        86
weighted avg    0.95396   0.95349   0.95316        86



## Resultados

Ambos modelos melhoraram sua performace/métricas ao fazer o ajuste de parâmetros. O melhor modelo obtido foi o XGBoost com os parâmetros: 

{'n_estimators': 165, 'max_depth': 9, 'learning_rate': 0.08949784952325804, 'subsample': 0.6176988962630369, 'colsample_bytree': 0.5540031541801593}

## Implantando modelo

Para implantar o modelo foi criado um servidor em flask com API REST, para acessar basta executar o script abaixo em um arquivo.py e executar um POST request para http://localhost:5000/predict passando um json com os dados a serem previstos

In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

best_model = joblib.load("xgb.joblib")

decoder = joblib.load("encoder.joblib")

ordem_features = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
    'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
    'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
    'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se',
    'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
    'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

@app.route('/predict', methods=['POST'])
def predict():
    linha = request.get_json()
    df = pd.DataFrame([linha])
    df = df[ordem_features]

    previsao = best_model.predict(df)
    return {"previsao": decoder.inverse_transform(previsao)[0]}

if __name__ == '__main__':
    app.run(debug=True)


In [62]:
#teste do servidor:
import requests

dados = {'compactness_se':0.04904, 
'symmetry_worst':0.4601, 
'fractal_dimension_se':0.006193, 
'concavity_mean':0.3001, 
'fractal_dimension_mean':0.07871, 
'area_se':153.4, 
'texture_worst':17.33, 
'perimeter_worst':184.6, 
'texture_mean':10.38, 
'compactness_mean':0.2776, 
'symmetry_mean':0.2419, 
'symmetry_se':0.03003, 
'smoothness_worst':0.1622, 
'concave points_se':0.01587, 
'concave points_worst':0.2654, 
'concave points_mean':0.1471, 
'area_worst':2019, 
'smoothness_mean':0.1184, 
'radius_worst':25.38, 
'area_mean':1001, 
'radius_se':1.095, 
'smoothness_se':0.006399, 
'fractal_dimension_worst':0.1189, 
'compactness_worst':0.6656, 
'perimeter_mean':122.8, 
'concavity_se':0.05373, 
'texture_se':0.9053, 
'concavity_worst':0.7119, 
'radius_mean': 17.99, 
'perimeter_se':8.589
}

response = requests.post("http://localhost:5000/predict", json=dados) 
response.json()


{'previsao': 'M'}

# Conclusão

Com essa atividade foi possível observar melhoria dos resultados quando feito os ajustes dos hiperparâmetros, neste caso o XGB se saiu melhor que o Random Forest. A API criada foi feita usando flask, a cópia do arquivo.py do servidor se encontra na penúltima célula. Com isso a atividade foi concluida com êxito